# NUERONCE foundational-recovery Colab journal

Trains the **`base_35m`** rung (34.4M params, fresh init -- shapes don't match
`chat_11m`, so there is no weight transfer) on a corpus-stack pull, then runs
**ForgeLoop SFT** on the already-cleaned `data/foundational_recovery_v3_1`
rows with the byte-identical system prompt the sealed gate uses, then scores
the result against this repo's own acceptance bar
(`docs/CODEX_HANDOFF.md`, `FOUNDATIONAL_GENERATION_RECOVERY.md`):

1. `base_35m` held-out bpb <= 1.5
2. choice-ranking accuracy beats chance by >=15 pts on >=3 MCQ subjects
3. phase2 inference suite >= 60% valid / non-echo / stop-terminated
4. 5/5 raw transcripts are grammatical English addressing the question
5. the **sealed 8-item proof gate**, run once, unmodified

**Ground rules carried over from the recovery doc -- do not violate these:**
- Never edit `scripts/eval_foundational_proof_gate.py`'s 8 sealed cases to
  make the gate pass. Run it once per checkpoint and record the number.
- The corpus mix below caps every single source near 25% of bytes (the
  77%-arithmetic-poisoning lesson in `docs/RESULTS.md`) -- code is one
  register among several, not the bulk of the corpus.
- `base_35m` is a **fresh-init** run. There is no upcycling path from
  `chat_11m` in this repo; `train_checkpoint.py` will refuse to `--resume`
  across a config mismatch.
- Everything here is resumable across Colab disconnects: checkpoints are
  written atomically, backed up to Drive every 30 minutes, and every
  training subprocess can be relaunched with `--resume`.

**100-hour GPU budget -- enforced by a `gpu_budget_*.json` file on Drive that
survives disconnects; launch cells refuse to start once their share is spent:**
- corpus build (one-time, cached to Drive): ~2h
- base pretraining: **68h** at `--seq 1024` (now matches SFT `--max-len 1024`;
  the old 192/1024 mismatch left positions 193-1024 untrained going into SFT)
- ForgeLoop SFT: **18h** (early-stopping patience will likely use less)
- evals + transcripts: ~2h; the remaining ~10h is disconnect/restart slack
- every launch is capped at **11h** so it fits inside one Colab session --
  when a session dies or the cap hits, just rerun the same launch cell next
  session: it resumes from the checkpoint and the budget file keeps the
  cross-session running total. Do not raise the caps mid-run; if a stage is
  out of budget the notebook will tell you and the honest move is to ship
  what the budget bought.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine for
`base_35m`). This notebook targets the `nueronce` package (the repo formerly
named `cfna` was renamed -- do not point this at old `cfna_*` branches or
checkpoints, they are not resume-compatible).


In [ ]:
# 1) Clone/reset to the live repository branch and install dependencies.
from pathlib import Path
import os, subprocess, sys

REPO = "LMMinier/nueronce"
BRANCH = "fix/foundational-generation-recovery"  # has the sealed-gate + recovery scripts merged in
REPO_DIR = Path("/content/nueronce") if Path("/content").exists() else Path.cwd()

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                    f"https://github.com/{REPO}.git", str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".",
                "datasets", "tqdm", "pytest"], check=True)

HEAD = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import torch, nueronce
print("branch:", BRANCH)
print("commit:", HEAD)
print("nueronce:", nueronce.__file__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0),
          "| mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError("Enable a GPU runtime before training (Runtime -> Change runtime type).")


In [ ]:
# 2) Safety gates before any fp16/AMP training + confirm the sealed-gate
#    pipeline tests (byte-prefix equality, no state leak, etc.) are green
#    on this exact commit before spending GPU hours on it.
subprocess.run([sys.executable, "-m", "pytest",
                "tests/test_gpu_amp.py",
                "tests/test_foundational_recovery_pipeline.py",
                "-q"], check=True)


In [ ]:
# 3) Mount Drive. Checkpoints/corpus are keyed to the commit + config, so a
#    stale cache from an older architecture can't silently get reused.
import json
from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
CKPT_ROOT = DRIVE / "nueronce_checkpoints"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

VERSION = f"base35m_{HEAD[:12]}"
CORPUS_TAG = f"{VERSION}_1g_v2"  # bumped: v1's SOURCES included 3 sources this repo can't
                                  # actually pull (wikimedia_dump/catalog_manual/pmc_oa_bulk
                                  # loaders have no extractor here) plus a gated one
                                  # (bigcode/the-stack-smol) -- v1's Drive cache is 2 sources,
                                  # badly imbalanced. Bump this suffix again if SOURCES changes.
CORPUS_DIR = Path(f"/content/corpus_large_{CORPUS_TAG}")
DRIVE_CORPUS = DRIVE / f"corpus_large_{CORPUS_TAG}"
# _s1024 in the name: seq went 192 -> 1024, and an old 192-trained checkpoint
# must NOT silently --resume into this run. Fresh key, fresh init.
BASE_CKPT = CKPT_ROOT / f"nueronce_base_35m_{VERSION}_s1024.pt"
BASE_BEST = BASE_CKPT.with_name(BASE_CKPT.stem + "_best.pt")
SFT_CKPT = CKPT_ROOT / f"nueronce_forgeloop_sft_{VERSION}.pt"
SFT_BEST = SFT_CKPT.with_name(SFT_CKPT.stem + "_best.pt")

# --- 100h GPU budget, tracked on Drive so it survives disconnects ---
TOTAL_BUDGET_MIN = 100 * 60
BASE_BUDGET_MIN  = 68 * 60    # base pretraining share
SFT_BUDGET_MIN   = 18 * 60    # ForgeLoop SFT share
SESSION_CAP_MIN  = 660        # max minutes any single launch runs (one Colab session)
BUDGET_FILE = CKPT_ROOT / f"gpu_budget_{VERSION}.json"

def load_budget():
    if BUDGET_FILE.exists():
        return json.loads(BUDGET_FILE.read_text(encoding="utf-8"))
    return {"base_minutes_used": 0.0, "sft_minutes_used": 0.0}

def save_budget(b):
    BUDGET_FILE.write_text(json.dumps(b, indent=2), encoding="utf-8")

def add_budget(kind, minutes):
    b = load_budget()
    b[kind + "_minutes_used"] = b.get(kind + "_minutes_used", 0.0) + minutes
    save_budget(b)
    return b

BASE_LOG = Path("/content/base_train.log")
SFT_LOG = Path("/content/sft_train.log")

SYSTEM_PROMPT_FILE = REPO_DIR / "runs/forgeloop/system_prompt.txt"
SFT_DATA_DIR = REPO_DIR / "data/foundational_recovery_v3_1"  # already cleaned + committed

print("version:", VERSION)
print("corpus dir:", CORPUS_DIR)
print("base checkpoint:", BASE_CKPT)
print("sft checkpoint:", SFT_CKPT)
print("system prompt:", SYSTEM_PROMPT_FILE.read_text(encoding="utf-8"))
_b = load_budget()
print(f"GPU budget used so far: base {_b['base_minutes_used']/60:.1f}h / {BASE_BUDGET_MIN/60:.0f}h | "
      f"sft {_b['sft_minutes_used']/60:.1f}h / {SFT_BUDGET_MIN/60:.0f}h | total cap 100h")


## 4) Build the rebalanced corpus

`the_stack_smol` (code) is capped at 20% of bytes here, not left to dominate
the mix like the old night-session run (452MB/907MB, ~50%) -- the repo's own
25%-per-source rule (`docs/LOCAL_TRAINING_PLAYBOOK.md`) exists precisely
because one source at 77% already poisoned an earlier SFT pass. Adjust
SOURCES below to change the mix; keep every value under ~25% of the total.


In [ ]:
# 4) Rebalanced multi-subject corpus. Each source is pulled into its own temp
#    directory (so dump_corpus_stack.py's per-run manifest.jsonl don't clobber
#    each other), then merged into one corpus_large/ manifest.
import json, shutil

REBUILD_CORPUS = False  # True forces a rebuild even if Drive has this version cached

# Every entry here MUST have loader == "huggingface" and must not be a
# gated dataset -- english_wikipedia_latest/project_gutenberg/pmc_oa_comm use
# loaders (wikimedia_dump/catalog_manual/pmc_oa_bulk) with no extractor
# implemented in dump_corpus_stack.py (silently skipped, not downloaded), and
# the_stack_smol (bigcode/the-stack-smol) is gated and needs a manual
# per-account click-through on the dataset page even with a token set. Both
# failure modes look identical downstream: fewer bytes than requested, and
# the remaining sources silently take over a bigger share than intended.
_ALLOWED_LOADERS = {"huggingface"}
_GATED_SOURCE_IDS = {"the_stack_smol"}  # bigcode/the-stack-smol requires manual HF access approval

SOURCES = {                                 # source_id -> target bytes
    "cosmopedia_100k":          200_000_000,  # 20% -- synthetic educational prose
    "open_web_math":            150_000_000,  # 15% -- math
    "fineweb_edu_sample_10bt":  200_000_000,  # 20% -- general/encyclopedic web text
    "smollm_python_edu":        200_000_000,  # 20% -- ungated code (bigcode/the-stack-smol is gated)
    "smollm_fineweb_edu_dedup": 150_000_000,  # 15% -- distinct general web text
    "tinystories":              100_000_000,  # 10% -- additional diversity; swapped in for
                                               # smollm_cosmopedia_v2, which crashed (SIGABRT)
                                               # on the actual run -- tinystories is the
                                               # simplest/most-tested source in this stack
                                               # (dump_corpus_stack.py's own docstring: "starts
                                               # with the user's requested stack: TinyStories
                                               # first"), lower risk of repeating that crash on
                                               # a future rebuild. Does not affect the corpus
                                               # already cached on Drive under CORPUS_TAG above --
                                               # only matters if REBUILD_CORPUS=True.
}   # total 1 GB, no source over 20% -- sized for the 68h base run; 500 MB
    # would be re-read for too many epochs and start memorizing

from nueronce.corpus.stack import get_entry as _get_entry
for _sid in SOURCES:
    _entry = _get_entry(_sid)
    if _entry.loader not in _ALLOWED_LOADERS:
        raise ValueError(f"{_sid}: loader={_entry.loader!r} has no HF extractor in this repo -- pick a different source")
    if _sid in _GATED_SOURCE_IDS:
        raise ValueError(f"{_sid}: gated dataset, needs manual HF access approval -- pick a different source")

def _reconstruct_manifest_from_files(corpus_dir):
    """Rebuild manifest.jsonl by scanning stack_text/*.txt when the manifest
    itself is missing/empty but the underlying files are intact. Recovers
    from a manifest-only corruption (e.g. an earlier bug wrote an empty
    record list back to Drive while the actual downloaded text was never
    touched) without re-downloading anything."""
    from datetime import date
    from hashlib import sha256
    from nueronce.corpus.stack import get_entry
    stack_text = corpus_dir / "stack_text"
    if not stack_text.is_dir():
        return []
    records = []
    today = date.today().isoformat()
    for f in sorted(stack_text.glob("*.txt")):
        stem = f.stem  # "{source_id}__{split}"
        if "__" not in stem:
            continue
        source_id, split = stem.rsplit("__", 1)
        if split not in ("train", "val"):
            continue
        try:
            entry = get_entry(source_id)
        except Exception:
            continue
        data = f.read_bytes()
        if not data:
            continue
        digest = sha256(data).hexdigest()
        records.append({
            "document_id": f"{entry.source_id}_{split}",
            "title": f"{entry.name} ({split})",
            "author": "dataset",
            "document_type": entry.role,
            "source_collection": entry.name,
            "source_locator": entry.dataset_page,
            "files_page": entry.files_page,
            "license": entry.license,
            "license_id": entry.license,
            "commercial_use": "noncommercial" not in entry.license.lower(),
            "attribution_required": any(t in entry.license.lower() for t in ("by", "sharing", "share")),
            "language": "en",
            "publication_year": None,
            "retrieved_at": today,
            "content_hash": f"sha256:{digest}",
            "quality_score": 1.0,
            "n_bytes": len(data),
            "n_docs": 1,
            "split": split,
            "bucket": f"phase_{entry.phase}_{entry.role}",
            "phase": entry.phase,
            "role": entry.role,
            "path": str(f.relative_to(corpus_dir)),
        })
    return records


if not REBUILD_CORPUS and DRIVE_CORPUS.exists() and (DRIVE_CORPUS / "manifest.jsonl").exists():
    if CORPUS_DIR.exists():
        shutil.rmtree(CORPUS_DIR)
    shutil.copytree(DRIVE_CORPUS, CORPUS_DIR)
    print("restored corpus from Drive:", CORPUS_DIR)

    manifest_path = CORPUS_DIR / "manifest.jsonl"
    existing = [json.loads(l) for l in manifest_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    if not existing:
        print("restored manifest.jsonl is EMPTY -- reconstructing it from the files on disk "
              "(no re-download needed, only the index was corrupted)")
        rebuilt = _reconstruct_manifest_from_files(CORPUS_DIR)
        if rebuilt:
            with manifest_path.open("w", encoding="utf-8") as fh:
                for rec in rebuilt:
                    fh.write(json.dumps(rec) + "\n")
            with (DRIVE_CORPUS / "manifest.jsonl").open("w", encoding="utf-8") as fh:
                for rec in rebuilt:
                    fh.write(json.dumps(rec) + "\n")
            print(f"reconstructed {len(rebuilt)} manifest records from stack_text/*.txt "
                  f"and repaired the Drive copy so this doesn't recur")
        else:
            print("no recoverable .txt files found either -- REBUILD_CORPUS = True is the only way forward")
else:
    # Sweep stale HF filelock files first -- if an earlier forced-stop killed
    # a download mid-flight, it left a lock behind that nothing will ever
    # release, and the next attempt to pull that dataset just hangs forever
    # waiting to acquire it (not slow -- deadlocked). Safe to remove: we are
    # about to start fresh downloads, nothing valid depends on a currently
    # held lock from a dead process.
    import glob
    hf_cache = Path.home() / ".cache" / "huggingface"
    stale_locks = glob.glob(str(hf_cache / "**" / "*.lock"), recursive=True)
    for lock_path in stale_locks:
        try:
            Path(lock_path).unlink()
        except OSError:
            pass
    if stale_locks:
        print(f"removed {len(stale_locks)} stale HF lock file(s) before starting")

    if CORPUS_DIR.exists():
        shutil.rmtree(CORPUS_DIR)
    (CORPUS_DIR / "stack_text").mkdir(parents=True, exist_ok=True)
    merged_records = []
    failed_sources = []
    PER_SOURCE_TIMEOUT_SECONDS = 1200  # 20 min -- generous for a real slow pull, but a hang must not eat the whole session
    for source_id, target_bytes in SOURCES.items():
        tmp_dir = Path(f"/content/_corpus_tmp_{source_id}")
        if tmp_dir.exists():
            shutil.rmtree(tmp_dir)
        cmd = [sys.executable, "-u", str(REPO_DIR / "scripts/dump_corpus_stack.py"),
               "--out", str(tmp_dir), "--sources", source_id,
               "--target-bytes", str(target_bytes), "--val-every", "20"]
        print(" ".join(cmd))
        # check=False + timeout: one source failing OR hanging (network
        # hiccup, HF outage, a stale lock from an earlier forced-stop, a
        # future gating change) must not silently skew the mix or freeze the
        # cell forever -- log it loudly and keep going.
        try:
            result = subprocess.run(cmd, cwd=REPO_DIR, timeout=PER_SOURCE_TIMEOUT_SECONDS)
            returncode = result.returncode
        except subprocess.TimeoutExpired:
            returncode = None
            print(f"  !! {source_id} did not finish within {PER_SOURCE_TIMEOUT_SECONDS}s -- "
                  f"treating as hung/stuck and skipping it")
        manifest_path = tmp_dir / "manifest.jsonl"
        source_records = []
        if manifest_path.exists():
            source_records = [json.loads(l) for l in manifest_path.read_text(encoding="utf-8").splitlines() if l.strip()]
        if returncode != 0 or not source_records:
            failed_sources.append(source_id)
            if returncode is not None:
                print(f"  !! {source_id} produced 0 usable records (exit code {returncode}) -- skipping it")
            shutil.rmtree(tmp_dir, ignore_errors=True)
            continue
        for f in (tmp_dir / "stack_text").glob("*"):
            shutil.copy2(f, CORPUS_DIR / "stack_text" / f.name)
        merged_records += source_records
        got_bytes = sum(r.get("n_bytes", 0) for r in source_records)
        print(f"  {source_id}: {got_bytes/1e6:.1f} MB / {target_bytes/1e6:.0f} MB requested")
        shutil.rmtree(tmp_dir)

    if failed_sources:
        print(f"\nWARNING: these sources produced nothing and were skipped: {failed_sources}")
        print("The corpus below is built only from the sources that actually worked --")
        print("check the per-source share in the next cell before trusting the 25% cap.")

    with (CORPUS_DIR / "manifest.jsonl").open("w", encoding="utf-8") as fh:
        for rec in merged_records:
            fh.write(json.dumps(rec) + "\n")

    if DRIVE_CORPUS.exists():
        shutil.rmtree(DRIVE_CORPUS)
    shutil.copytree(CORPUS_DIR, DRIVE_CORPUS)
    print("built + saved corpus to Drive:", DRIVE_CORPUS)


In [ ]:
# 5) Corpus audit + auto-rebalance (exact one-shot).
#
# dump_corpus_stack.py writes ONE file per (source, split), and
# nueronce.corpus.dataset.ByteCorpus reads each file's entire contents by
# path (it ignores the manifest's n_bytes) -- so rebalancing must truncate
# the actual train .txt on disk; editing metadata alone would print pretty
# numbers while training silently used the full, imbalanced files.
#
# The final per-source size is computed by water-filling in closed form
# (not by iterating truncate-and-recompute, which converges only
# asymptotically): find the smallest set of k largest sources to clamp such
# that x = CAP * U / (1 - CAP*k) is consistent -- every clamped source
# starts >= x and every unclamped source is <= x -- then truncate each
# clamped source's TRAIN file (never val: that protects the held-out bpb
# signal) so the source totals exactly x. One truncation pass, exact result.
from collections import Counter, defaultdict

CAP = 0.25

def _load_records():
    return [json.loads(l) for l in (CORPUS_DIR / "manifest.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]

def _source_totals(records):
    b = defaultdict(int)
    for r in records:
        b[r["source_collection"]] += int(r.get("n_bytes", 0))
    return b, sum(b.values())

def _print_shares(records, label):
    source_bytes, total = _source_totals(records)
    print(f"-- {label}: {len(records)} records, {total/1e6:.1f} MB --")
    for name, n in sorted(source_bytes.items(), key=lambda kv: -kv[1]):
        share = n / total if total else 0.0
        flag = "  <-- ABOVE 25% CAP" if share > CAP else ""
        print(f"{name:30s} {n/1e6:7.1f} MB  {share*100:5.1f}%{flag}")
    return source_bytes, total

records = _load_records()
assert records, "corpus manifest is empty -- rerun cell 4; its repair path rebuilds the manifest from files on disk"

splits = Counter(r["split"] for r in records)
assert splits["val"] >= 2, "need at least 2 held-out documents for a real bpb signal"

source_bytes, total_bytes = _print_shares(records, "before rebalance")

# --- water-filling: pick k = number of clamped (largest) sources ---
sizes = sorted(source_bytes.items(), key=lambda kv: -kv[1])  # descending
n_sources = len(sizes)
solution = None
for k in range(0, n_sources):
    if 1.0 - CAP * k <= 0:
        break  # cap unsatisfiable with this many clamped sources
    unclamped_total = sum(b for _, b in sizes[k:])
    x = CAP * unclamped_total / (1.0 - CAP * k)
    ok_clamped = all(b >= x - 1 for _, b in sizes[:k])
    ok_unclamped = all(b <= x + 1 for _, b in sizes[k:])
    if ok_clamped and ok_unclamped:
        solution = (k, int(x))
        break

changed_paths = []
if solution is None:
    print("cannot rebalance: too few sources survived the build to satisfy the cap --")
    print("add another source to SOURCES and rerun the build cell (the final assert below will fail)")
elif solution[0] > 0:
    k, x = solution
    clamped_names = [name for name, _ in sizes[:k]]
    print(f"one-shot target: clamp {k} source(s) {clamped_names} to {x/1e6:.1f} MB each")
    for name in clamped_names:
        val_bytes = sum(r.get("n_bytes", 0) for r in records
                        if r["source_collection"] == name and r["split"] != "train")
        remaining = max(0, x - val_bytes)
        train_recs = sorted((r for r in records
                             if r["source_collection"] == name and r["split"] == "train"),
                            key=lambda r: -r.get("n_bytes", 0))
        for r in train_recs:
            cur = int(r.get("n_bytes", 0))
            keep = min(cur, remaining)
            remaining -= keep
            if keep < cur:
                fp = CORPUS_DIR / r["path"]
                if fp.exists():
                    data = fp.read_bytes()
                    fp.write_bytes(data[:keep])
                    changed_paths.append(r["path"])
                    print(f"  truncated {name} train file {cur/1e6:.1f} MB -> {keep/1e6:.1f} MB")
                r["n_bytes"] = keep

    with (CORPUS_DIR / "manifest.jsonl").open("w", encoding="utf-8") as fh:
        for rec in records:
            fh.write(json.dumps(rec) + "\n")
    if DRIVE_CORPUS.exists():
        # push back ONLY what changed (truncated files + manifest), so a
        # future "restore from Drive" can't resurrect untruncated files
        for rel in changed_paths:
            dst_file = DRIVE_CORPUS / rel
            dst_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(CORPUS_DIR / rel, dst_file)
        with (DRIVE_CORPUS / "manifest.jsonl").open("w", encoding="utf-8") as fh:
            for rec in records:
                fh.write(json.dumps(rec) + "\n")
        print(f"pushed {len(changed_paths)} truncated file(s) + manifest back to Drive")
else:
    print("all sources already within the cap -- nothing to do")

records = _load_records()
splits = Counter(r["split"] for r in records)
source_bytes, total_bytes = _print_shares(records, "after rebalance")

assert splits["val"] >= 2, "need at least 2 held-out documents for a real bpb signal"
# 1e-3 headroom: integer flooring can leave a share a hair off exactly 25.000%
assert all(n / total_bytes <= CAP + 1e-3 for n in source_bytes.values()), \
    "rebalance could not satisfy the cap -- too few sources; add another source and rerun the build cell"


## 6) Base pretraining -- `base_35m`, fresh init

Runs as a background process. 6c is the main loop -- it polls, prints
progress, backs up to Drive, and stops cleanly (SIGINT) once
`BASE_TRAIN_MINUTES` is spent, so this is safe to leave in a "Run All"
-- it is bounded and does not need manual babysitting. 6a/6b still work as
optional on-demand peeks (rerun anytime) if you're driving the notebook by
hand instead; 6b is worth checking manually for the repetition-collapse
failure signature already documented on this architecture. Note:
`train_checkpoint.py` keeps only the *latest* snapshot, so 6c snapshots its
own `*_best.pt` copy the moment the held-out bpb improves (approximate to
one 5-minute poll) -- a late-run regression no longer loses the best base
model, and SFT warm-starts from that best copy, not from wherever the clock
happened to stop.


In [ ]:
# 6) Launch base pretraining as a background process.
import time

_b = load_budget()
BASE_REMAINING = max(0.0, BASE_BUDGET_MIN - _b["base_minutes_used"])
BASE_TRAIN_MINUTES = int(min(SESSION_CAP_MIN, BASE_REMAINING))
print(f"base budget: {_b['base_minutes_used']:.0f} min used / {BASE_BUDGET_MIN} min "
      f"-> this session: {BASE_TRAIN_MINUTES} min")
if BASE_TRAIN_MINUTES <= 0:
    raise RuntimeError("base pretraining budget (68h) is spent. Move on to SFT/eval -- "
                       "raising BASE_BUDGET_MIN in cell 3 is a conscious decision, not a reflex.")
# rerun this cell (same BASE_CKPT) each new session to continue

def checkpoint_is_valid(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        try:
            ck = torch.load(path, map_location="cpu", weights_only=True)
        except Exception:
            # the config object may be a custom class the safe loader rejects;
            # this is our own Drive checkpoint, so fall back rather than
            # misclassifying it as invalid and torching the resume.
            ck = torch.load(path, map_location="cpu", weights_only=False)
        return "state_dict" in ck and "config" in ck
    except Exception as exc:
        print("checkpoint failed to load, will fresh-init instead:", repr(exc))
        return False

base_cmd = [sys.executable, "-u", str(REPO_DIR / "scripts/train_checkpoint.py"),
            "--preset", "base_35m", "--corpus", str(CORPUS_DIR),
            "--minutes", str(BASE_TRAIN_MINUTES), "--seq", "1024", "--batch", "8",
            "--lr", "3e-4", "--amp", "--device", "cuda", "--out", str(BASE_CKPT)]
if checkpoint_is_valid(BASE_CKPT):
    base_cmd.append("--resume")

log_fh = open(BASE_LOG, "a")
# start_new_session=True detaches the trainer from the notebook kernel's
# process group: Colab's cell-interrupt sends SIGINT to the whole group, and
# without this, stopping ANY unrelated cell kills the background training as
# collateral (observed: KeyboardInterrupt mid-backward, exit code 1). The
# deliberate budget-cap stop in 6c still works -- send_signal targets the
# trainer's PID directly, not the group.
BASE_PROC = subprocess.Popen(base_cmd, cwd=REPO_DIR, stdout=log_fh,
                             stderr=subprocess.STDOUT, start_new_session=True)
print("launched base training, PID", BASE_PROC.pid)
print("log:", BASE_LOG, "| checkpoint:", BASE_CKPT)


In [ ]:
# 6a) Monitor the background run (rerun any time).
import re
alive = BASE_PROC.poll() is None
tail = BASE_LOG.read_text(errors="replace").splitlines()[-15:] if BASE_LOG.exists() else []
print("training alive:", alive, "" if alive else f"(exit code {BASE_PROC.returncode})")
print("\n".join(tail))
bpbs = [float(m.group(1)) for l in BASE_LOG.read_text(errors="replace").splitlines()
        if (m := re.search(r"held-out bpb ([0-9.]+)", l))]
if bpbs:
    print(f"\nheld-out bpb: start {bpbs[0]:.3f} -> now {bpbs[-1]:.3f} (best {min(bpbs):.3f}; target <= 1.5)")


In [ ]:
# 6b) Live inference probe -- loads the checkpoint on CPU so it never
#     touches the GPU mid-step; atomic writes make reads always consistent.
#     Watch specifically for the documented failure signature: repetition
#     loops ("the state of the state of the...") or collapse to whitespace.
from nueronce.chat import load_checkpoint

if not BASE_CKPT.exists():
    print("no checkpoint yet")
else:
    model, _ = load_checkpoint(str(BASE_CKPT))
    model = model.to("cpu").eval()

    @torch.no_grad()
    def complete_bytes(prompt: str, max_new: int = 100, max_ctx: int = 256) -> str:
        ids = list(prompt.encode("utf-8"))
        out = bytearray()
        for _ in range(max_new):
            ctx = torch.tensor([ids[-max_ctx:]], dtype=torch.long)
            logits, _ = model(ctx)
            nxt = int(logits[0, -1].argmax())
            ids.append(nxt)
            out.append(nxt)
        return out.decode("utf-8", errors="replace")

    for p in ["The nature of human understanding is", "Once upon a time", "def add(a, b):\n"]:
        print(f">>> {p!r}")
        print(complete_bytes(p))
        print()


In [ ]:
# 6c) Poll until base training finishes or BASE_TRAIN_MINUTES is spent --
#     Run-All safe: prints progress and backs up to Drive automatically, no
#     manual re-running of 6a/6b required. Sends SIGINT (not SIGKILL) on
#     timeout so the trainer exits through its normal save path; falls back
#     to terminate() only if it doesn't stop within the grace period.
import re, signal

POLL_SECONDS = 300
BACKUP_SECONDS = 1800
STOP_GRACE_SECONDS = 60

start = time.time()
last_backup = start
last_budget_mark = start
best_seen = float("inf")
while True:
    try:
        rc = BASE_PROC.wait(timeout=POLL_SECONDS)
        print(f"base training finished on its own, exit code {rc}")
        break
    except subprocess.TimeoutExpired:
        pass

    elapsed_min = (time.time() - start) / 60
    log_text = BASE_LOG.read_text(errors="replace") if BASE_LOG.exists() else ""
    bpbs = [float(m.group(1)) for l in log_text.splitlines() if (m := re.search(r"held-out bpb ([0-9.]+)", l))]
    trend = f"best {min(bpbs):.3f}, last {bpbs[-1]:.3f}" if bpbs else "no evals logged yet"
    print(f"[{elapsed_min:6.1f} min] {trend}")

    # the base trainer only keeps the *latest* snapshot -- keep our own copy
    # of the best-bpb checkpoint (approximate to one poll interval)
    if bpbs and min(bpbs) < best_seen and BASE_CKPT.exists():
        best_seen = min(bpbs)
        shutil.copy2(BASE_CKPT, BASE_BEST)
        print(f"    new best held-out bpb {best_seen:.3f} -> snapshotted {BASE_BEST.name}")

    now = time.time()
    add_budget("base", (now - last_budget_mark) / 60)
    last_budget_mark = now

    # (no separate Drive backup step: BASE_CKPT/BASE_BEST are written straight
    # to Drive already -- copying a file onto itself raises SameFileError and
    # would have crashed this loop at the 30-minute mark)

    if elapsed_min >= BASE_TRAIN_MINUTES:
        print(f"time budget ({BASE_TRAIN_MINUTES} min) reached -- stopping training cleanly")
        BASE_PROC.send_signal(signal.SIGINT)
        try:
            rc = BASE_PROC.wait(timeout=STOP_GRACE_SECONDS)
            print(f"stopped cleanly, exit code {rc}")
        except subprocess.TimeoutExpired:
            BASE_PROC.terminate()
            rc = BASE_PROC.wait(timeout=STOP_GRACE_SECONDS)
            print(f"forced stop after grace period, exit code {rc}")
        break

add_budget("base", (time.time() - last_budget_mark) / 60)
if not BASE_CKPT.exists():
    _tail = BASE_LOG.read_text(errors="replace").splitlines()[-25:] if BASE_LOG.exists() else []
    raise RuntimeError("base training exited without writing a checkpoint -- last log lines:\n" + "\n".join(_tail))
_b = load_budget()
print(f"base GPU minutes used so far: {_b['base_minutes_used']:.0f} / {BASE_BUDGET_MIN}")
base_ck = torch.load(BASE_BEST if BASE_BEST.exists() else BASE_CKPT, map_location="cpu", weights_only=False)
history = base_ck.get("history", [])
best_bpb = min((h["heldout_bpb"] for h in history), default=float("nan"))
print(f"\nstep {base_ck.get('step')} | best held-out bpb {best_bpb:.4f} | target <= 1.5: "
      f"{'PASS' if best_bpb <= 1.5 else 'not yet -- rerun cell 6 to keep training'}")


## 6.5) Scale up the SFT set

`data/foundational_recovery_v3_1` alone is 3,684 rows total (2,574 train) --
`docs/RESULTS.md` names this exact dataset by row count as too small to
generalize to unseen phrasing on the sealed gate: *"3,684 SFT rows on ~14 MB
of pretraining cannot provide"* that. Per `docs/CODEX_HANDOFF.md`'s fix, mix
in `nueronce.training.synthetic_dialogue` (~127K template-generated pairs)
and `nueronce.training.mcq_sft` (ARC/CommonsenseQA/MathQA/GSM8K -- **OpenBookQA
is held out of SFT entirely** so 8a can score it as a pure transfer check),
capped per category/subject -- raw `synthetic_dialogue` output is 77.6%
arithmetic before capping, the exact poisoning shape `docs/RESULTS.md`
already warns about, measured below rather than assumed away. The original
v3.1 `val.jsonl`/`test.jsonl` stay untouched as the held-out set -- only
`train.jsonl` grows, and the SFT launch cell below samples domains uniformly
(`--balanced-domain-sampling`) rather than relying on row-count ratios.
Every candidate train row is also checked (case/whitespace-normalized)
against the v3.1 `val.jsonl`/`test.jsonl` prompts and dropped on a match, so
template-generated data cannot leak held-out phrasing into training.


In [ ]:
# 6.5) Build the scaled-up SFT training file: the v3.1 ForgeLoop rows kept
#      whole, plus capped synthetic_dialogue and capped MCQ subjects. val/test
#      are left as the original v3.1 files -- nothing new leaks into
#      validation, and gate_ready / response_argmax_acc stay comparable to
#      the small-data run.
import hashlib
import random
from collections import Counter

from nueronce.training.synthetic_dialogue import generate_all
from nueronce.training.mcq_sft import load_and_convert
from nueronce.corpus.stack import get_entry

SYNTHETIC_CATEGORY_CAP = 8000
MCQ_SUBJECT_CAP = 8000
# openbookqa is deliberately HELD OUT of SFT -- cell 8a scores it as a pure
# transfer check (knowledge without format-fit). Do not add it back here.
MCQ_SUBJECTS_FOR_SFT = ["arc_easy", "arc_challenge", "commonsense_qa", "math_qa", "gsm8k"]
SCALED_TRAIN = SFT_DATA_DIR.parent / "foundational_recovery_v3_1_scaled" / "train.jsonl"
SCALED_TRAIN.parent.mkdir(parents=True, exist_ok=True)


def _record(prompt, response, domain):
    return {"prompt": prompt, "response": response, "domain": domain}


def _norm(text: str) -> str:
    # normalization for leak-checking: case- and whitespace-insensitive
    return " ".join(text.lower().split())

# never let a train row collide with the held-out v3.1 val/test prompts --
# an exact OR normalized match counts as a leak and is dropped.
heldout_prompts = set()
for split_name in ("val.jsonl", "test.jsonl"):
    split_path = SFT_DATA_DIR / split_name
    if not split_path.exists():
        continue
    for line in split_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        heldout_prompts.add(_norm(rec["prompt"]))
print(f"leak filter: {len(heldout_prompts)} held-out val/test prompts loaded")

leaked = 0
seen = set()
rows = []

# a) the existing cleaned ForgeLoop curriculum, kept whole -- this is the
#    software-engineering/ForgeLoop-specific coverage the sealed gate itself
#    draws its 8 cases from the same distribution as.
for line in (SFT_DATA_DIR / "train.jsonl").read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    row = dict(json.loads(line))
    row["domain"] = "forgeloop"
    if _norm(row["prompt"]) in heldout_prompts:
        leaked += 1
        continue
    key = hashlib.sha256((row["prompt"] + "|" + row["response"]).encode("utf-8")).hexdigest()
    if key not in seen:
        seen.add(key)
        rows.append(row)
print(f"forgeloop v3.1: {len(rows)} rows")

# b) synthetic_dialogue, capped per category.
by_category = {}
for rec in generate_all():
    by_category.setdefault(rec["category"], []).append(rec)
for category, recs in sorted(by_category.items()):
    added = 0
    for rec in recs[:SYNTHETIC_CATEGORY_CAP]:
        prompt, response = rec["messages"][0]["content"], rec["messages"][1]["content"]
        if _norm(prompt) in heldout_prompts:
            leaked += 1
            continue
        key = hashlib.sha256((prompt + "|" + response).encode("utf-8")).hexdigest()
        if key in seen:
            continue
        seen.add(key)
        rows.append(_record(prompt, response, f"synthetic_{category}"))
        added += 1
    print(f"synthetic_dialogue/{category:22s} {added:6d} / {len(recs):6d} available (cap {SYNTHETIC_CATEGORY_CAP})")

# c) MCQ choice-ranking + free-form QA, capped per subject.
for subject in MCQ_SUBJECTS_FOR_SFT:
    recs = list(load_and_convert(get_entry(subject), split="train", limit=MCQ_SUBJECT_CAP))
    added = 0
    for rec in recs:
        prompt, response = rec["messages"][0]["content"], rec["messages"][1]["content"]
        if _norm(prompt) in heldout_prompts:
            leaked += 1
            continue
        key = hashlib.sha256((prompt + "|" + response).encode("utf-8")).hexdigest()
        if key in seen:
            continue
        seen.add(key)
        rows.append(_record(prompt, response, f"mcq_{subject}"))
        added += 1
    print(f"mcq_sft/{subject:18s} {added:6d} rows (cap {MCQ_SUBJECT_CAP})")

random.Random(1234).shuffle(rows)
with SCALED_TRAIN.open("w", encoding="utf-8") as fh:
    for row in rows:
        fh.write(json.dumps(row) + "\n")

domain_counts = Counter(row["domain"] for row in rows)
print(f"\nleak filter dropped {leaked} rows matching a held-out val/test prompt")
print(f"total unique rows: {len(rows)}  ->  {SCALED_TRAIN}")
for domain, n in domain_counts.most_common():
    share = n / len(rows)
    flag = "  <-- ABOVE 25% CAP" if share > 0.25 else ""
    print(f"  {domain:24s} {n:6d}  {share*100:5.1f}%{flag}")


## 7) ForgeLoop SFT on the cleaned v3.1 rows

Uses the scaled `train.jsonl` built in 6.5 (v3.1 ForgeLoop rows + capped
synthetic_dialogue + capped MCQ subjects), the original v3.1 `val.jsonl` as
held-out validation, and the
**exact same system-prompt file** the sealed gate evaluates with
(`runs/forgeloop/system_prompt.txt`) -- fix-ladder step 3, satisfied by
construction rather than by convention. `train_forgeloop_sft.py` already has
safe-interruption checkpointing, atomic saves, and a separate best.pt built
in (`docs/RESULTS.md` fix-ladder step 1: it also reports
`response_argmax_acc` and `first8_loss`, and flags `gate_ready` once val loss
<= 0.05 -- gate attempts below that are known to be a waste of a run).
7b polls and stops cleanly at `SFT_TRAIN_MINUTES` the same way 6c does for
base pretraining, so this stage is also "Run All"-safe and bounded.


In [ ]:
# 7) Launch ForgeLoop SFT as a background process, warm-started from the
#    base_35m checkpoint (or resumed from its own prior progress if SFT_CKPT
#    already exists).
_b = load_budget()
SFT_REMAINING = max(0.0, SFT_BUDGET_MIN - _b["sft_minutes_used"])
SFT_TRAIN_MINUTES = int(min(SESSION_CAP_MIN, SFT_REMAINING))  # enforced by the wait() timeout below
print(f"sft budget: {_b['sft_minutes_used']:.0f} min used / {SFT_BUDGET_MIN} min "
      f"-> this session: {SFT_TRAIN_MINUTES} min")
if SFT_TRAIN_MINUTES <= 0:
    raise RuntimeError("SFT budget (18h) is spent -- go run the eval cells on the best checkpoint.")

BASE_FOR_SFT = BASE_BEST if BASE_BEST.exists() else BASE_CKPT
print("warm-starting SFT from:", BASE_FOR_SFT.name)

sft_cmd = [sys.executable, "-u", str(REPO_DIR / "scripts/train_forgeloop_sft.py"),
           "--base", str(BASE_FOR_SFT), "--train", str(SCALED_TRAIN),
           "--val", str(SFT_DATA_DIR / "val.jsonl"), "--out", str(SFT_CKPT),
           "--system-file", str(SYSTEM_PROMPT_FILE),
           "--batch", "8", "--max-len", "1024", "--lr", "5e-5",
           "--eval-every", "25", "--checkpoint-every", "25",
           "--max-steps", "100000", "--patience", "40", "--balanced-domain-sampling"]

sft_log_fh = open(SFT_LOG, "a")
# start_new_session=True: same notebook-interrupt isolation as the base
# trainer launch -- see the comment there.
SFT_PROC = subprocess.Popen(sft_cmd, cwd=REPO_DIR, stdout=sft_log_fh,
                            stderr=subprocess.STDOUT, start_new_session=True)
print("launched ForgeLoop SFT, PID", SFT_PROC.pid)
print("log:", SFT_LOG, "| checkpoint:", SFT_CKPT, "| best:", SFT_BEST)


In [ ]:
# 7a) Monitor SFT (rerun any time) -- parses the JSON records the trainer
#     already prints per eval, including the gate_ready flag.
alive = SFT_PROC.poll() is None
print("training alive:", alive, "" if alive else f"(exit code {SFT_PROC.returncode})")
records = []
for line in SFT_LOG.read_text(errors="replace").splitlines():
    line = line.strip()
    if line.startswith("{"):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            pass
evals = [r for r in records if "val_loss" in r]
if evals:
    last = evals[-1]
    print(f"step {last['sft_step']} | val_loss {last['val_loss']:.4f} | "
          f"response_argmax_acc {last['response_argmax_acc']:.3f} | "
          f"first8_loss {last['first8_loss']:.4f} | "
          f"gate_ready {last['gate_ready']} | best_val {last['best_val_loss']:.4f}")
else:
    print("no eval rows yet")


In [ ]:
# 7b) Poll until SFT converges, hits max-steps, or SFT_TRAIN_MINUTES is spent
#     -- Run-All safe: prints progress and backs up to Drive automatically.
#     Sends SIGINT on timeout; the trainer's KeyboardInterrupt handler saves
#     before exiting, so this is a clean stop, not a kill.
import signal

POLL_SECONDS = 300
BACKUP_SECONDS = 1800
STOP_GRACE_SECONDS = 60

start = time.time()
last_backup = start
last_budget_mark = start
while True:
    try:
        rc = SFT_PROC.wait(timeout=POLL_SECONDS)
        print(f"SFT finished on its own, exit code {rc}")
        break
    except subprocess.TimeoutExpired:
        pass

    elapsed_min = (time.time() - start) / 60
    records = []
    for line in SFT_LOG.read_text(errors="replace").splitlines():
        line = line.strip()
        if line.startswith("{"):
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    evals = [r for r in records if "val_loss" in r]
    if evals:
        last = evals[-1]
        print(f"[{elapsed_min:6.1f} min] step {last['sft_step']} | val_loss {last['val_loss']:.4f} | "
              f"response_argmax_acc {last['response_argmax_acc']:.3f} | "
              f"gate_ready {last['gate_ready']} | best_val {last['best_val_loss']:.4f}")
    else:
        print(f"[{elapsed_min:6.1f} min] no eval rows yet")

    now = time.time()
    add_budget("sft", (now - last_budget_mark) / 60)
    last_budget_mark = now

    # (no separate Drive backup step: SFT_CKPT/SFT_BEST are written straight
    # to Drive already -- self-copy raises SameFileError)

    if elapsed_min >= SFT_TRAIN_MINUTES:
        print(f"time budget ({SFT_TRAIN_MINUTES} min) reached -- stopping SFT cleanly")
        SFT_PROC.send_signal(signal.SIGINT)
        try:
            rc = SFT_PROC.wait(timeout=STOP_GRACE_SECONDS)
            print(f"stopped cleanly, exit code {rc}")
        except subprocess.TimeoutExpired:
            SFT_PROC.terminate()
            rc = SFT_PROC.wait(timeout=STOP_GRACE_SECONDS)
            print(f"forced stop after grace period, exit code {rc}")
        break

add_budget("sft", (time.time() - last_budget_mark) / 60)
_b = load_budget()
print(f"sft GPU minutes used so far: {_b['sft_minutes_used']:.0f} / {SFT_BUDGET_MIN}")

EVAL_CKPT = SFT_BEST if SFT_BEST.exists() else SFT_CKPT
if not EVAL_CKPT.exists():
    _tail = SFT_LOG.read_text(errors="replace").splitlines()[-25:] if SFT_LOG.exists() else []
    raise RuntimeError("SFT exited without writing a checkpoint -- last log lines:\n" + "\n".join(_tail))
sft_ck = torch.load(EVAL_CKPT, map_location="cpu", weights_only=False)
print(f"\nevaluating from: {EVAL_CKPT}")
print(f"best_val_loss: {sft_ck.get('best_val_loss'):.4f} (gate-ready threshold: 0.05)")


## 8) Score against the acceptance bar

Four checks that don't just re-measure aggregate loss, plus the sealed gate.
**Run the proof-gate cell exactly once per checkpoint and record the number
-- do not iterate against it.** If it fails, the next move is back to step 7
(more SFT / data), never editing the gate's 8 cases.


In [ ]:
# 8a) Choice-ranking MCQ accuracy vs. chance (measures knowledge even while
#     generation is still imperfect -- structure-before-content, per
#     docs/RESULTS.md).
from nueronce.chat import load_checkpoint
from nueronce.corpus.stack import get_entry
from nueronce.training.mcq_sft import load_and_convert, evaluate_mcq

model, _ = load_checkpoint(str(EVAL_CKPT))
model = model.to("cpu").eval()

MCQ_TRAINED = ["arc_easy", "arc_challenge", "commonsense_qa", "math_qa", "gsm8k"]  # SFT saw these
MCQ_SUBJECTS = MCQ_TRAINED + ["openbookqa"]  # openbookqa = pure transfer, never in SFT
mcq_results = {}
for subject in MCQ_SUBJECTS:
    records = list(load_and_convert(get_entry(subject), split="val", limit=200))
    if not records:
        records = list(load_and_convert(get_entry(subject), split="test", limit=200))
    if not records:
        print(f"{subject}: no held-out records available, skipping")
        continue
    result = evaluate_mcq(model, records, max_examples=100)
    mcq_results[subject] = result
    beats_chance = (result["accuracy"] - result["chance"]) * 100
    tag = "  [TRANSFER -- never in SFT]" if subject not in MCQ_TRAINED else ""
    print(f"{subject:18s} acc {result['accuracy']:.3f}  chance {result['chance']:.3f}  "
          f"beats-chance {beats_chance:+.1f} pts  n={result['n']}{tag}")

n_beating_15 = sum(1 for s in MCQ_TRAINED
                   if s in mcq_results
                   and (mcq_results[s]["accuracy"] - mcq_results[s]["chance"]) * 100 >= 15)
transfer = mcq_results.get("openbookqa")
if transfer:
    print(f"transfer check (openbookqa, never in SFT): "
          f"{(transfer['accuracy'] - transfer['chance']) * 100:+.1f} pts vs chance "
          f"-- solidly positive means knowledge, not just format-fit")
print(f"\nsubjects beating chance by >=15 pts: {n_beating_15}/3 required -- "
      f"{'PASS' if n_beating_15 >= 3 else 'NOT YET'}")

Path("metrics").mkdir(exist_ok=True)
Path("metrics/colab_mcq_results.json").write_text(json.dumps(mcq_results, indent=2), encoding="utf-8")


In [ ]:
# 8b) Phase2 generative inference suite (>= 60% valid / non-echo /
#     stop-terminated is the acceptance bar).
subprocess.run([sys.executable, "-u", str(REPO_DIR / "scripts/eval_inference_phase2.py"),
                "--checkpoint", str(EVAL_CKPT),
                "--out", "metrics/colab_inference_phase2.json"],
               cwd=REPO_DIR, check=True)
print(json.dumps(json.loads(Path("metrics/colab_inference_phase2.json").read_text(encoding="utf-8")),
                  indent=2)[:2000])


In [ ]:
# 8c) THE SEALED PROOF GATE. Run once per checkpoint, unmodified. Do not
#     rerun this repeatedly against the same checkpoint hoping for a
#     different number, and never edit the 8 cases in
#     scripts/eval_foundational_proof_gate.py to make this pass.
subprocess.run([sys.executable, "-u", str(REPO_DIR / "scripts/eval_foundational_proof_gate.py"),
                "--checkpoint", str(EVAL_CKPT),
                "--system-file", str(SYSTEM_PROMPT_FILE),
                "--output", "metrics/colab_foundational_proof_gate.json",
                "--no-fail-exit"],
               cwd=REPO_DIR, check=True)
gate = json.loads(Path("metrics/colab_foundational_proof_gate.json").read_text(encoding="utf-8"))
print(json.dumps(gate, indent=2)[:3000])


In [ ]:
# 8d) 5 raw transcripts, unedited -- the fluency bar (grammatical English
#     addressing the question), not the correctness bar.
from nueronce.chat import Conversation

conversation = Conversation(model, system=SYSTEM_PROMPT_FILE.read_text(encoding="utf-8"),
                             temperature=0.2, max_new=160, max_ctx=384)

for prompt in [
    "Hello. Introduce yourself in one sentence.",
    "What is 17 plus 26?",
    "Summarize the plan you would follow to fix a failing unit test.",
    "Rewrite this politely: 'send me the file now'.",
    "What should you do if you are not sure an answer is correct?",
]:
    print(">>>", prompt)
    print(conversation.say(prompt))
    print()


## 9) Acceptance scorecard + persist


In [ ]:
# 9) Consolidated scorecard from everything written to metrics/ above.
_base_src = BASE_BEST if BASE_BEST.exists() else BASE_CKPT
base_history = torch.load(_base_src, map_location="cpu", weights_only=False).get("history", [])
base_bpb = min((h["heldout_bpb"] for h in base_history), default=float("nan"))
sft_val_loss = sft_ck.get("best_val_loss", float("nan"))
phase2 = json.loads(Path("metrics/colab_inference_phase2.json").read_text(encoding="utf-8"))
gate = json.loads(Path("metrics/colab_foundational_proof_gate.json").read_text(encoding="utf-8"))

print("=== ACCEPTANCE SCORECARD ===")
print(f"1) base_35m held-out bpb <= 1.5      : {base_bpb:.4f}  {'PASS' if base_bpb <= 1.5 else 'FAIL'}")
print(f"2) MCQ subjects beating chance >=15pt : {n_beating_15}/3  {'PASS' if n_beating_15 >= 3 else 'FAIL'}")
print(f"3) SFT response val_loss <= 0.05      : {sft_val_loss:.4f}  {'PASS' if sft_val_loss <= 0.05 else 'FAIL'}")
print("4) phase2 suite (see report above)    : inspect metrics/colab_inference_phase2.json for the pass rate")
print(f"5) sealed proof gate                  : {gate.get('overall', gate)}")
_b = load_budget()
print(f"\nGPU budget: base {_b['base_minutes_used']/60:.1f}h / {BASE_BUDGET_MIN/60:.0f}h | "
      f"sft {_b['sft_minutes_used']/60:.1f}h / {SFT_BUDGET_MIN/60:.0f}h | total cap 100h")
print()
print("A generalization dev set (>=100 unseen atomic examples, distinct from")
print("the training templates) is still owed per the advancement rules --")
print("data/foundational_val32 is the existing precedent but is sized for a")
print("probe, not the full requirement; build it out before declaring this done.")


In [ ]:
# 10) Persist everything to Drive. Git push is NOT automatic -- review the
#     diff yourself before pushing metrics/checkpoints from a Colab session.
# checkpoints already live in CKPT_ROOT on Drive -- nothing to copy
# (a self-copy would raise SameFileError)
for name in ["colab_mcq_results.json", "colab_inference_phase2.json", "colab_foundational_proof_gate.json"]:
    src = Path("metrics") / name
    if src.exists():
        shutil.copy2(src, DRIVE / "nueronce_checkpoints" / name)
print("backed up to", CKPT_ROOT)

print(f'''
To push results back (run manually, review the diff first):
  cd {REPO_DIR}
  git add metrics/colab_*.json
  git commit -m "colab: base_35m + forgeloop SFT run {VERSION}"
  git push origin {BRANCH}
''')
